In [4]:
from sliced_hierarchical_OT import compute_hierarchical_wasserstein, compute_wasserstein_distance
from sliced_hierarchical_OT import calc_SHOT_between_subsets, calc_HOT_between_subsets
from point_cloud_utils import *
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
from plotting_utils import plot_means_with_errorbars


def create_overview_dict(itermax, param_ls):
    dict_overview = {"HOT": {}, "SHOT": {}, "OT_NNA": {}}
    for param in param_ls:
        for key in dict_overview.keys():
            dict_overview[key][param] = []
            
    return dict_overview

def create_ablation_dict(itermax, param_ls):
    dict_overview = {"HOT": {}, "SHOT1": {}, "SHOT2": {}, "SHOT3": {}}
    for param in param_ls:
        for key in dict_overview.keys():
            dict_overview[key][param] = []
            
    return dict_overview


def eval_overview_dict(dict_overview):
    hot_means = pd.DataFrame.from_dict(dict_overview["HOT"]).mean().values
    hot_stds = pd.DataFrame.from_dict(dict_overview["HOT"]).std().values
    shot_means = pd.DataFrame.from_dict(dict_overview["SHOT"]).mean().values
    shot_stds = pd.DataFrame.from_dict(dict_overview["SHOT"]).std().values
    nna_means = pd.DataFrame.from_dict(dict_overview["OT_NNA"]).mean().values
    nna_stds = pd.DataFrame.from_dict(dict_overview["OT_NNA"]).std().values
    return hot_means, hot_stds, shot_means, shot_stds, nna_means, nna_stds

def eval_ablation_dict(dict_overview):
    hot_means = pd.DataFrame.from_dict(dict_overview["HOT"]).mean().values
    hot_stds = pd.DataFrame.from_dict(dict_overview["HOT"]).std().values
    shot_means1 = pd.DataFrame.from_dict(dict_overview["SHOT1"]).mean().values
    shot_stds1 = pd.DataFrame.from_dict(dict_overview["SHOT1"]).std().values
    shot_means2 = pd.DataFrame.from_dict(dict_overview["SHOT2"]).mean().values
    shot_stds2 = pd.DataFrame.from_dict(dict_overview["SHOT2"]).std().values
    shot_means3 = pd.DataFrame.from_dict(dict_overview["SHOT3"]).mean().values
    shot_stds3 = pd.DataFrame.from_dict(dict_overview["SHOT3"]).std().values
    return hot_means, hot_stds, shot_means1, shot_stds1, shot_means2, shot_stds2, shot_means3, shot_stds3


modelnet_base_path = "/homes/numerik/piening/scratch/RandomWalkonGraph_OT/SFTLB/data/ModelNet10"
save_folder = "PointCloud_Viz/"

# Shape Subsampling

In [5]:
np.random.seed(42)
torch.manual_seed(42)

num_ls = list(range(1, 20, 1))
itermax = 5
dict_overview_num = create_overview_dict(itermax, num_ls)

for iternum in range(itermax):        
    dataset = ModelNet10(modelnet_base_path, selected_class="bed", max_shapes=10, split="train", seed=iternum)
    for num in tqdm(num_ls):
        dataset2 = ModelNet10(modelnet_base_path, selected_class="bed", max_shapes=num, std=0., split="test", seed=iternum)
        hot = calc_HOT_between_subsets(dataset, dataset2).item()
        dict_overview_num["HOT"][num].append(hot)
        ot_nna = calc_OT_NNA_between_subsets(dataset, dataset2).item()
        dict_overview_num["OT_NNA"][num].append(ot_nna)
        shot = calc_SHOT_between_subsets(dataset, dataset2, length_scale=0.1, outer_pnum=100, grid_n=10, inner_pnum=100).item()
        dict_overview_num["SHOT"][num].append(shot)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 19/19 [00:32<00:00,  1.69s/it]


In [14]:
hot_means, hot_stds, shot_means, shot_stds, nna_means, nna_stds = eval_overview_dict(dict_overview_num)

plot_means_with_errorbars(num_ls, hot_means, hot_stds, xlabel="# Shapes in Target", 
                          ylabel="WoW", save_path=save_folder + "wow_num_shapes.png", x_target=10, legend=False, ystrnum=5)

plot_means_with_errorbars(num_ls, shot_means, shot_stds, xlabel="# Shapes in Target", 
                          ylabel="Ours", save_path=save_folder + "swow_num_shapes.png", x_target=10, legend=False, ystrnum=5)

plot_means_with_errorbars(num_ls, nna_means, nna_stds, xlabel="# Shapes in Target", 
                          ylabel="OT-NNA", save_path=save_folder + "ot_nna_num_shapes.png", x_target=10, ystrnum=5)

# Gaussian Noise

In [7]:
np.random.seed(42)
torch.manual_seed(42)

std_ls = np.linspace(0., .5, 10)
itermax = 5
dict_overview_std = create_overview_dict(itermax, std_ls)

for iternum in range(itermax):        
    dataset = ModelNet10(modelnet_base_path, selected_class="sofa", max_shapes=10, split="train", seed=iternum)
    for std in tqdm(std_ls):
        dataset2 = ModelNet10(modelnet_base_path, selected_class="sofa", max_shapes=10, std=std, split="test", seed=iternum)
        hot = calc_HOT_between_subsets(dataset, dataset2).item()
        dict_overview_std["HOT"][std].append(hot)
        shot = calc_SHOT_between_subsets(dataset, dataset2, length_scale=.1, outer_pnum=100, grid_n=10, inner_pnum=100).item()
        dict_overview_std["SHOT"][std].append(shot)
        ot_nna = calc_OT_NNA_between_subsets(dataset, dataset2).item()
        dict_overview_std["OT_NNA"][std].append(ot_nna)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:12<00:00,  1.21s/it]


In [13]:
hot_means, hot_stds, shot_means, shot_stds, nna_means, nna_stds = eval_overview_dict(dict_overview_std)

plot_means_with_errorbars(std_ls, hot_means, hot_stds, xlabel="Gaussian Noise in Target", 
                          ylabel="WoW", save_path=save_folder + "wow_gaussian.png", legend=False, ystrnum=5)

plot_means_with_errorbars(std_ls, shot_means, shot_stds, xlabel="Gaussian Noise in Target", 
                          ylabel="Ours", save_path=save_folder + "swow_gaussian.png", legend=False, ystrnum=5)

plot_means_with_errorbars(std_ls, nna_means, nna_stds, xlabel="Gaussian Noise in Target", 
                          ylabel="OT-NNA", save_path=save_folder + "ot_nna_gaussian.png", legend=False, ystrnum=5)


# Point Cloud Downsampling

In [9]:
np.random.seed(42)
torch.manual_seed(42)

down_ls = [i * 50 for i in range(1, 13)]
itermax = 5
dict_overview_down = create_overview_dict(itermax, down_ls)

for iternum in range(itermax):        
    dataset = ModelNet10(modelnet_base_path, selected_class="chair", max_shapes=10, split="train", seed=iternum, n_points=500)
    for down in tqdm(down_ls):
        dataset2 = ModelNet10(modelnet_base_path, selected_class="chair", max_shapes=10, std=0., split="test", seed=iternum,
                             n_points=down)
        hot = calc_HOT_between_subsets(dataset, dataset2).item()
        dict_overview_down["HOT"][down].append(hot)
        shot = calc_SHOT_between_subsets(dataset, dataset2, length_scale=.1, outer_pnum=100, grid_n=10, inner_pnum=100).item()
        dict_overview_down["SHOT"][down].append(shot)
        ot_nna = calc_OT_NNA_between_subsets(dataset, dataset2).item()
        dict_overview_down["OT_NNA"][down].append(ot_nna)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [01:55<00:00,  9.66s/it]


In [15]:
hot_means, hot_stds, shot_means, shot_stds, nna_means, nna_stds = eval_overview_dict(dict_overview_down)

plot_means_with_errorbars(down_ls, hot_means, hot_stds, xlabel="# Points in Target", 
                          ylabel="WoW", save_path=save_folder + "wow_num_points.png", x_target=500, legend=False, ystrnum=5)
plot_means_with_errorbars(down_ls, shot_means, shot_stds, xlabel="# Points in Target", 
                          ylabel="Ours", save_path=save_folder + "swow_num_points.png", x_target=500, legend=False, ystrnum=5)
plot_means_with_errorbars(down_ls, nna_means, nna_stds, xlabel="# Points in Target",
                          ylabel="OT-NNA", save_path=save_folder + "ot_nna_num_points.png", x_target=500, legend=False, ystrnum=5)


# Timing

In [16]:
from time import perf_counter
dataset = ModelNet10(modelnet_base_path, selected_class="monitor", max_shapes=10, split="train", seed=iternum, n_points=500)
dataset2 = ModelNet10(modelnet_base_path, selected_class="monitor", max_shapes=10, std=0., split="test", seed=iternum, n_points=500)
t0 = perf_counter()
hot = calc_HOT_between_subsets(dataset, dataset2).item()
print("HOT took: ", perf_counter() - t0)
t0 = perf_counter()
shot = calc_SHOT_between_subsets(dataset, dataset2, length_scale=.1, outer_pnum=100, grid_n=10, inner_pnum=100).item()
print("SHOT took: ", perf_counter() - t0)
t0 = perf_counter()
ot_nna = calc_OT_NNA_between_subsets(dataset, dataset2).item()
print("OT-NNA took: ", perf_counter() - t0)

HOT took:  4.044683831045404
SHOT took:  0.25238097715191543
OT-NNA took:  7.680671571055427


# Ablation Number of Points

In [18]:
np.random.seed(42)
torch.manual_seed(42)

down_ls = [i * 100 for i in range(1, 11)]
itermax = 5
dict_overview_down_ablation = create_ablation_dict(itermax, down_ls)
dict_overview_down_time = create_ablation_dict(itermax, down_ls)


for iternum in range(itermax):        
    for down in tqdm(down_ls):
        dataset = ModelNet10(modelnet_base_path, selected_class="chair", max_shapes=10, split="train", seed=iternum, n_points=down)
        dataset2 = ModelNet10(modelnet_base_path, selected_class="chair", max_shapes=10, std=0., split="test", seed=iternum,
                             n_points=down)
        t0 = perf_counter()
        hot = calc_HOT_between_subsets(dataset, dataset2).item()
        time_took_hot = perf_counter() - t0
        t0 = perf_counter()
        dict_overview_down_ablation["HOT"][down].append(hot)
        dict_overview_down_time["HOT"][down].append(time_took_hot)
        shot = calc_SHOT_between_subsets(dataset, dataset2, length_scale=.1, outer_pnum=100, grid_n=10, inner_pnum=10).item()
        time_took_shot = perf_counter() - t0
        dict_overview_down_time["SHOT1"][down].append(time_took_shot)
        dict_overview_down_ablation["SHOT1"][down].append(shot)
        shot = calc_SHOT_between_subsets(dataset, dataset2, length_scale=.1, outer_pnum=100, grid_n=10, inner_pnum=100).item()
        time_took_shot = perf_counter() - t0
        dict_overview_down_time["SHOT2"][down].append(time_took_shot)
        dict_overview_down_ablation["SHOT2"][down].append(shot)
        shot = calc_SHOT_between_subsets(dataset, dataset2, length_scale=.1, outer_pnum=100, grid_n=10, inner_pnum=500).item()
        time_took_shot = perf_counter() - t0
        dict_overview_down_time["SHOT3"][down].append(time_took_shot)
        dict_overview_down_ablation["SHOT3"][down].append(shot)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [01:16<00:00,  7.70s/it]


In [19]:
hot_means, hot_stds, shot_means1, shot_stds1, shot_means2, shot_stds2, shot_means3, shot_stds3 = eval_ablation_dict(dict_overview_down_time)

plot_means_with_errorbars(down_ls, hot_means, hot_stds, xlabel="# Points", 
                          ylabel="Time (WoW)", save_path=save_folder + "outer_ablation/time_wow_num_points.png", 
                          x_target=None, legend=False, 
                         ylim = (0., 1.2 * hot_means.max()), ystrnum=5)

plot_means_with_errorbars(down_ls, shot_means1, shot_stds1, xlabel="# Points", 
                          ylabel="Time (S=1000)", save_path=save_folder + "outer_ablation/time_swow1_num_points.png", 
                          x_target=None, legend=False,
                         ylim = (0., 1.2 * hot_means.max()), ystrnum=5)

plot_means_with_errorbars(down_ls, shot_means2, shot_stds2, xlabel="# Points", 
                          ylabel="Time (S=10000)", save_path=save_folder + "outer_ablation/time_swow2_num_points.png", 
                          x_target=None, legend=False,
                         ylim = (0., 1.2 * hot_means.max()), ystrnum=5)

plot_means_with_errorbars(down_ls, shot_means3, shot_stds3, xlabel="# Points", 
                          ylabel="Time (S=50000)", save_path=save_folder + "outer_ablation/time_swow3_num_points.png", 
                          x_target=None, legend=False,
                         ylim = (0., 1.2 * hot_means.max()), ystrnum=5)

hot_means, hot_stds, shot_means1, shot_stds1, shot_means2, shot_stds2, shot_means3, shot_stds3 = eval_ablation_dict(dict_overview_down_ablation)

plot_means_with_errorbars(down_ls, hot_means, hot_stds, xlabel="# Points", 
                          ylabel="WoW", save_path=save_folder + "outer_ablation/values_wow_num_points.png", x_target=None, 
                          legend=True, ystrnum=5)

plot_means_with_errorbars(down_ls, shot_means1, shot_stds1, xlabel="# Points", 
                          ylabel="Ours (S=1000)", save_path=save_folder + "outer_ablation/values_swow1_num_points.png", 
                          x_target=None, legend=False, ystrnum=5)

plot_means_with_errorbars(down_ls, shot_means2, shot_stds2, xlabel="# Points", 
                          ylabel="Ours (S=10000)", save_path=save_folder + "outer_ablation/values_swow2_num_points.png", 
                          x_target=None, legend=False, ystrnum=5)

plot_means_with_errorbars(down_ls, shot_means3, shot_stds3, xlabel="# Points", 
                          ylabel="Ours (S=50000)", save_path=save_folder + "outer_ablation/values_swow3_num_points.png", 
                          x_target=None, legend=False, ystrnum=5)


# Ablation Number of Shapes

In [20]:
np.random.seed(42)
torch.manual_seed(42)

snum_ls = [10 * i for i in range(1, 11)]
itermax = 5
dict_overview_snum_ablation = create_ablation_dict(itermax, snum_ls)
dict_overview_snum_time = create_ablation_dict(itermax, snum_ls)
down = 50


for iternum in range(itermax):    
    for snum in tqdm(snum_ls):
        dataset = ModelNet10(modelnet_base_path, selected_class="chair", max_shapes=snum, split="train", seed=iternum, n_points=down)
        dataset2 = ModelNet10(modelnet_base_path, selected_class="chair", max_shapes=snum, std=0., split="test", seed=iternum,
                             n_points=down)
        t0 = perf_counter()
        hot = calc_HOT_between_subsets(dataset, dataset2).item()
        time_took_hot = perf_counter() - t0
        t0 = perf_counter()
        dict_overview_snum_ablation["HOT"][snum].append(hot)
        dict_overview_snum_time["HOT"][snum].append(time_took_hot)
        shot = calc_SHOT_between_subsets(dataset, dataset2, length_scale=.1, outer_pnum=10, grid_n=10, inner_pnum=10).item()
        time_took_shot = perf_counter() - t0
        dict_overview_snum_time["SHOT1"][snum].append(time_took_shot)
        dict_overview_snum_ablation["SHOT1"][snum].append(shot)
        shot = calc_SHOT_between_subsets(dataset, dataset2, length_scale=.1, outer_pnum=100, grid_n=10, inner_pnum=10).item()
        time_took_shot = perf_counter() - t0
        dict_overview_snum_time["SHOT2"][snum].append(time_took_shot)
        dict_overview_snum_ablation["SHOT2"][snum].append(shot)
        shot = calc_SHOT_between_subsets(dataset, dataset2, length_scale=.1, outer_pnum=500, grid_n=10, inner_pnum=10).item()
        time_took_shot = perf_counter() - t0
        dict_overview_snum_time["SHOT3"][snum].append(time_took_shot)
        dict_overview_snum_ablation["SHOT3"][snum].append(shot)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [03:57<00:00, 23.74s/it]


In [21]:
hot_means, hot_stds, shot_means1, shot_stds1, shot_means2, shot_stds2, shot_means3, shot_stds3 = eval_ablation_dict(dict_overview_snum_time)

plot_means_with_errorbars(snum_ls, hot_means, hot_stds, xlabel="# Shapes", 
                          ylabel="Time (WoW)", save_path=save_folder + "inner_ablation/time_wow_num_shapes.png", 
                          x_target=None, legend=False, 
                         ylim = (0., 1.2 * hot_means.max() ), ystrnum=5)


plot_means_with_errorbars(snum_ls, shot_means1, shot_stds1, xlabel="# Shapes", 
                          ylabel="Time (S=100)", save_path=save_folder + "inner_ablation/time_swow1_num_shapes.png", 
                          x_target=None, legend=False,
                         ylim = (0., 1.2 * hot_means.max()), ystrnum=5)


plot_means_with_errorbars(snum_ls, shot_means2, shot_stds2, xlabel="# Shapes", 
                          ylabel="Time (S=1000)", save_path=save_folder + "inner_ablation/time_swow2_num_shapes.png", 
                          x_target=None, legend=False,
                         ylim = (0., 1.2 * hot_means.max()), ystrnum=5)


plot_means_with_errorbars(snum_ls, shot_means3, shot_stds3, xlabel="# Shapes", 
                          ylabel="Time (S=5000)", save_path=save_folder + "inner_ablation/time_swow3_num_shapes.png", 
                          x_target=None, legend=False,
                         ylim = (0., 1.2 * hot_means.max()), ystrnum=5)


hot_means, hot_stds, shot_means1, shot_stds1, shot_means2, shot_stds2, shot_means3, shot_stds3 = eval_ablation_dict(dict_overview_snum_ablation)

plot_means_with_errorbars(snum_ls, hot_means, hot_stds, xlabel="# Shapes", 
                          ylabel="WoW", save_path=save_folder + "inner_ablation/values_wow_num_shapes.png", 
                          x_target=None, legend=True, ystrnum=5)


plot_means_with_errorbars(snum_ls, shot_means1, shot_stds1, xlabel="# Shapes", 
                          ylabel="Ours (S=100)", save_path=save_folder + "inner_ablation/values_swow1_num_shapes.png", 
                          x_target=None, legend=False, ystrnum=5)


plot_means_with_errorbars(snum_ls, shot_means2, shot_stds2, xlabel="# Shapes", 
                          ylabel="Ours (S=1000)", save_path=save_folder + "inner_ablation/values_swow2_num_shapes.png", 
                          x_target=None, legend=False, ystrnum=5)


plot_means_with_errorbars(snum_ls, shot_means3, shot_stds3, xlabel="# Shapes", 
                          ylabel="Ours (S=5000)", save_path=save_folder + "inner_ablation/values_swow3_num_shapes.png", 
                          x_target=None, legend=False, ystrnum=5)
